# Experiment 3 — Real-Time Temporal Degradation

**Objective:** Quantify how tracking quality degrades when edge devices skip frames
due to limited inference throughput. A device running at *k* FPS on a 30 FPS feed
processes roughly one frame in every 30/*k*. The tracker never sees the skipped
frames, so IoU overlap shrinks, Kalman predictions accumulate error, and identity
associations degrade.

**Method:** Strided replay of existing per-frame detection caches — no new
inference required. For each stride level, supervision ByteTrack is re-run on
only the selected frames to produce realistic track associations. Ground truth
is filtered to the same strided frame set before metric computation.

**Detection sources:**
- FP32 desktop cache → applies to all CPU-only devices (RPi 5 CPU, RPi 4, Jetson Nano)
- Hailo INT8 cache → applies exclusively to RPi 5 Hailo rows
- Jetson TRT HQ cache → applies exclusively to Jetson Nano TRT rows

**Effect decomposition:**
- (A) Resolution — already characterised in Experiment 1
- (B) Temporal on FP32 — strided replay of desktop detections (NEW)
- (C) Quantisation — Hailo INT8 stride=1 vs FP32 stride=1 (NEW)
- (D) Temporal on INT8 — strided replay of Hailo detections (NEW)

In [ ]:
# ── Configuration ─────────────────────────────────────────────────────────────
import os
import multiprocessing as mp
from concurrent.futures import ProcessPoolExecutor
from pathlib import Path

import numpy as np
import pandas as pd

from benchmark.config import DATA_ROOT, RESULTS_RAW, SEQUENCES, SEQ_SUFFIX
from benchmark.mot_gt import load_gt

# Model variants evaluated on edge devices (N / S / M only — L and X excluded)
MODELS = ["yolo26n", "yolo26s", "yolo26m"]
RESOLUTIONS = [640, 576]

# Representative FPS levels and corresponding strides from a 30 FPS source.
# Each device maps to its nearest tested level.
FPS_LEVELS = {30: 1, 15: 2, 10: 3, 7: 4, 5: 6, 3: 10, 2: 15, 1: 30}

# Device → per-variant (actual_fps_at_640, detection_source)
# detection_source: 'fp32' uses desktop FP32 cache; 'int8' uses Hailo INT8 cache;
#                   'trt' uses Jetson Nano TensorRT HQ cache
DEVICE_MAP = {
    "rpi5_cpu":        {"N": (5.2,  "fp32"), "S": (2.0,  "fp32"), "M": (0.8,  "fp32")},
    "rpi5_hailo":      {"N": (51.9, "int8"), "S": (34.0, "int8"), "M": (19.5, "int8")},
    "rpi4":            {"N": (1.3,  "fp32"), "S": (0.6,  "fp32"), "M": (0.2,  "fp32")},
    "jetson_nano":     {"N": (7.5,  "fp32"), "S": (4.3,  "fp32"), "M": (2.0,  "fp32")},
    "jetson_nano_trt": {"N": (13.4, "trt"),  "S": (7.6,  "trt"),  "M": (3.6,  "trt")},
    "arduino_uno_q":   {"N": (2.5,  "fp32"), "S": (1.0,  "fp32"), "M": (0.3,  "fp32")},
}

# Profiles to include in the paper-facing outputs. Edit this list to add or
# remove devices without deleting profiling result files.
PROFILES = {
    "rpi5_cpu": {
        "label": "RPi 5 CPU",
        "table_tex": r"\makecell[l]{\\RPi\,5\\CPU}",
        "marker": "o",
        "color": "#1f77b4",
    },
    "rpi5_hailo": {
        "label": "RPi 5 NPU",
        "table_tex": r"\makecell[l]{\\RPi\,5\\NPU\tnote{$\dagger$}}",
        "marker": "D",
        "color": "#ff7f0e",
    },
    "rpi4": {
        "label": "RPi 4",
        "table_tex": r"\makecell[l]{\\RPi\,4\\CPU}",
        "marker": "s",
        "color": "#2ca02c",
    },
    "jetson_nano": {
        "label": "Jetson Nano",
        "table_tex": r"\makecell[l]{\\Jetson\\Nano\\CUDA}",
        "marker": "^",
        "color": "#d62728",
    },
    "jetson_nano_trt": {
        "label": "Jetson Nano TRT",
        "table_tex": r"\makecell[l]{\\Jetson\\Nano\\TRT}",
        "marker": "v",
        "color": "#8c564b",
    },
    "arduino_uno_q": {
        "label": "Arduino Uno Q",
        "table_tex": r"\makecell[l]{\\Arduino\\Uno\,Q}",
        "marker": "P",
        "color": "#7f7f7f",
    },
}

# Output directory for temporal degradation results
OUT_DIR = RESULTS_RAW.parent / "temporal"
OUT_DIR.mkdir(parents=True, exist_ok=True)

DEFAULT_N_WORKERS = min(24, max(1, (os.cpu_count() or 1) - 1))
N_WORKERS = max(1, int(os.environ.get("TEMPORAL_N_WORKERS", DEFAULT_N_WORKERS)))
MP_START_METHOD = "fork" if "fork" in mp.get_all_start_methods() else None
USE_PARALLEL = N_WORKERS > 1 and MP_START_METHOD is not None


def assign_fps_level(actual_fps: float) -> int:
    """Map actual device FPS to the nearest representative level."""
    levels = sorted(FPS_LEVELS.keys())
    return min(levels, key=lambda x: abs(x - actual_fps))


def get_track_buffer(stride: int, mode: str = "scaled") -> int:
    """Track buffer preserving ~1-second reacquisition window.

    Scaled mode: buffer shrinks proportionally with stride so the semantic
    window stays constant (~1 s). Fixed mode: always 30 (sensitivity check).
    """
    if mode == "scaled":
        return max(5, round(30 / stride))
    return 30


# Variant letter → model stem mapping
_VARIANT_LETTER = {"N": "yolo26n", "S": "yolo26s", "M": "yolo26m"}

print(f"Sequences   : {SEQUENCES}")
print(f"Models      : {MODELS}")
print(f"Resolutions : {RESOLUTIONS}")
print(f"Profiles    : {list(PROFILES.keys())}")
print(f"Workers     : {N_WORKERS} ({'parallel' if USE_PARALLEL else 'serial'})")
print(f"FPS levels  : {list(FPS_LEVELS.keys())}")
print(f"Output dir  : {OUT_DIR}")

In [ ]:
# ── GT loading ────────────────────────────────────────────────────────────────
# Load and cache ground-truth annotations for all three sequences.
# GT track counts are needed for metric normalisation (IDSW/GT-track, MT ratio).

gt_cache = {}     # seq → DataFrame[frame_id, track_id, x, y, w, h]
n_gt_tracks = {}  # seq → int

for seq in SEQUENCES:
    seq_dir = DATA_ROOT / f"{seq}-{SEQ_SUFFIX}"
    gt_df = load_gt(seq_dir)
    gt_cache[seq] = gt_df
    n_gt_tracks[seq] = gt_df["track_id"].nunique()
    print(f"{seq}: {len(gt_df):>6,} GT rows, {n_gt_tracks[seq]:>3} tracks, "
          f"frames {gt_df['frame_id'].min()}–{gt_df['frame_id'].max()}")

In [ ]:
# ── Detection cache loading ───────────────────────────────────────────────────
# Load all raw CSVs into a dict keyed by (seq, model, imgsz, precision).
# FP32 desktop CSVs: {seq}_{model}.pt_{imgsz}_desktop.csv
# Hailo INT8 CSVs:   {seq}_{model}_hq.hef_{imgsz}_rpi5_hailo.csv  (640)
#                    {seq}_{model}_{imgsz}_hq.hef_{imgsz}_rpi5_hailo.csv  (576)
# Jetson TRT CSVs:   {seq}_{model}_hq.engine_{imgsz}_jetson_nano_trt.csv  (640)
#                    {seq}_{model}_{imgsz}_hq.engine_{imgsz}_jetson_nano_trt.csv  (576)
import json

def load_det_cache_csv(path: Path) -> pd.DataFrame:
    """Load a raw detection CSV and pre-parse bbox/conf payloads once."""
    df = pd.read_csv(path)
    df["bboxes_arr"] = df["bboxes_xyxy"].map(lambda s: np.array(json.loads(s), dtype=np.float32))
    df["confs_arr"] = df["confs"].map(lambda s: np.array(json.loads(s), dtype=np.float32))
    return df

det_cache = {}  # (seq, model, imgsz, precision) → DataFrame
missing = []

for seq in SEQUENCES:
    for model in MODELS:
        for imgsz in RESOLUTIONS:
            # FP32 desktop cache
            fp32_csv = (RESULTS_RAW / "desktop" / "exp1" /
                        f"{seq}_{model}.pt_{imgsz}_desktop.csv")
            if fp32_csv.exists():
                det_cache[(seq, model, imgsz, "fp32")] = load_det_cache_csv(fp32_csv)
            else:
                missing.append(f"fp32: {fp32_csv.name}")

            # Hailo INT8 cache — filename encodes resolution in model name
            if imgsz == 640:
                hef_name = f"{model}_hq.hef"
            else:
                hef_name = f"{model}_{imgsz}_hq.hef"
            int8_csv = (RESULTS_RAW / "rpi5_hailo" / "exp1" /
                        f"{seq}_{hef_name}_{imgsz}_rpi5_hailo.csv")
            if int8_csv.exists():
                det_cache[(seq, model, imgsz, "int8")] = load_det_cache_csv(int8_csv)
            else:
                missing.append(f"int8: {int8_csv.name}")

            # Jetson Nano TRT HQ cache — filename mirrors Hailo-style baked resolutions
            if imgsz == 640:
                engine_name = f"{model}_hq.engine"
            else:
                engine_name = f"{model}_{imgsz}_hq.engine"
            trt_csv = (RESULTS_RAW / "jetson_nano_trt" / "exp1" /
                       f"{seq}_{engine_name}_{imgsz}_jetson_nano_trt.csv")
            if trt_csv.exists():
                det_cache[(seq, model, imgsz, "trt")] = load_det_cache_csv(trt_csv)
            else:
                missing.append(f"trt: {trt_csv.name}")

print(f"Loaded {len(det_cache)} detection caches")
if missing:
    print(f"Missing ({len(missing)}):")
    for m in missing:
        print(f"  {m}")
else:
    print("All expected caches present.")

# Expected: 3 seq × 3 models × 2 res × 3 sources = 54
expected_caches = len(SEQUENCES) * len(MODELS) * len(RESOLUTIONS) * 3
assert len(det_cache) == expected_caches, f"Expected {expected_caches}, got {len(det_cache)}"

In [ ]:
# ── strided_retrack() — re-run ByteTrack on strided frame subsets ─────────────
# Discards original track IDs (which assume stride=1) and feeds only the
# strided detections into a fresh supervision ByteTrack instance. Original
# MOT17 frame IDs are preserved in the saved predictions, while supervision
# ByteTrack advances one internal timestep per update_with_detections() call.
# Temporal semantics are therefore controlled by the replay cadence plus the
# chosen track_buffer, not by raw frame ID gaps in the output DataFrame.

import supervision as sv


def strided_retrack(
    raw_df: pd.DataFrame,
    stride: int,
    track_buffer: int = 30,
) -> tuple[pd.DataFrame, list[int]]:
    """Re-track detections at a given stride using supervision ByteTrack.

    Args:
        raw_df: Raw inference CSV (one row per frame) with columns
                frame_id, n_detections, bboxes_xyxy, confs.
        stride: Frame selection stride (1 = all frames, 30 = every 30th).
        track_buffer: ByteTrack lost_track_buffer parameter.

    Returns:
        (pred_df, strided_fids) where pred_df has columns
        [frame_id, track_id, x, y, w, h] in original frame IDs,
        and strided_fids is the list of original frame IDs selected.
    """
    # MOT17 frame IDs are 1-based, so select 1, 1+stride, 1+2*stride, ...
    selected = raw_df[(raw_df["frame_id"] - 1) % stride == 0].copy()
    strided_fids = selected["frame_id"].tolist()

    tracker = sv.ByteTrack(
        track_activation_threshold=0.25,
        lost_track_buffer=track_buffer,
        frame_rate=30,
    )

    rows = []
    for row in selected.itertuples(index=False):
        fid = int(row.frame_id)
        n_det = int(row.n_detections)

        if n_det == 0:
            # Advance tracker's internal frame counter on empty frames
            tracker.update_with_detections(sv.Detections.empty())
            continue

        bboxes = row.bboxes_arr if hasattr(row, "bboxes_arr") else np.array(json.loads(row.bboxes_xyxy), dtype=np.float32)
        confs = row.confs_arr if hasattr(row, "confs_arr") else np.array(json.loads(row.confs), dtype=np.float32)

        dets = sv.Detections(
            xyxy=bboxes,
            confidence=confs,
        )
        tracked = tracker.update_with_detections(dets)

        if tracked.tracker_id is None or len(tracked) == 0:
            continue

        # Convert xyxy → xywh for MOTChallenge accumulator compatibility
        for i in range(len(tracked)):
            x1, y1, x2, y2 = tracked.xyxy[i]
            rows.append({
                "frame_id": fid,
                "track_id": int(tracked.tracker_id[i]),
                "x": float(x1),
                "y": float(y1),
                "w": float(x2 - x1),
                "h": float(y2 - y1),
            })

    pred_df = pd.DataFrame(rows, columns=["frame_id", "track_id", "x", "y", "w", "h"])
    return pred_df, strided_fids


def select_frame_ids_for_fps(
    frame_ids: list[int],
    target_fps: int,
    source_fps: int = 30,
) -> list[int]:
    """Approximate arbitrary integer FPS from a fixed-FPS source cadence.

    Uses a Bresenham-style schedule so non-divisor targets such as 13 FPS or
    9 FPS alternate between short and long gaps while preserving the original
    MOT frame IDs.
    """
    target_fps = max(1, min(source_fps, int(target_fps)))
    if target_fps >= source_fps:
        return [int(fid) for fid in frame_ids]

    selected = []
    prev_slot = -1
    for fid in frame_ids:
        slot = int((int(fid) - 1) * target_fps / source_fps)
        if slot > prev_slot:
            selected.append(int(fid))
            prev_slot = slot
    return selected


def retrack_at_target_fps(
    raw_df: pd.DataFrame,
    target_fps: int,
    track_buffer: int = 30,
    source_fps: int = 30,
) -> tuple[pd.DataFrame, list[int]]:
    """Re-track detections using an approximate arbitrary target FPS cadence."""
    selected_fids = select_frame_ids_for_fps(
        raw_df["frame_id"].tolist(),
        target_fps=target_fps,
        source_fps=source_fps,
    )
    selected = raw_df[raw_df["frame_id"].isin(set(selected_fids))].copy()

    tracker = sv.ByteTrack(
        track_activation_threshold=0.25,
        lost_track_buffer=track_buffer,
        frame_rate=source_fps,
    )

    rows = []
    for row in selected.itertuples(index=False):
        fid = int(row.frame_id)
        n_det = int(row.n_detections)

        if n_det == 0:
            tracker.update_with_detections(sv.Detections.empty())
            continue

        bboxes = row.bboxes_arr if hasattr(row, "bboxes_arr") else np.array(json.loads(row.bboxes_xyxy), dtype=np.float32)
        confs = row.confs_arr if hasattr(row, "confs_arr") else np.array(json.loads(row.confs), dtype=np.float32)

        dets = sv.Detections(
            xyxy=bboxes,
            confidence=confs,
        )
        tracked = tracker.update_with_detections(dets)

        if tracked.tracker_id is None or len(tracked) == 0:
            continue

        for i in range(len(tracked)):
            x1, y1, x2, y2 = tracked.xyxy[i]
            rows.append({
                "frame_id": fid,
                "track_id": int(tracked.tracker_id[i]),
                "x": float(x1),
                "y": float(y1),
                "w": float(x2 - x1),
                "h": float(y2 - y1),
            })

    pred_df = pd.DataFrame(rows, columns=["frame_id", "track_id", "x", "y", "w", "h"])
    return pred_df, selected_fids


# Quick validation: stride=1 on a small cache should produce non-empty output
_test_key = next(iter(det_cache))
_test_pred, _test_fids = strided_retrack(det_cache[_test_key], stride=1)
print(f"Validation — {_test_key}: {len(_test_pred)} pred rows, "
      f"{len(_test_fids)} strided frames, "
      f"{_test_pred['track_id'].nunique()} unique tracks")

In [ ]:
# ── compute_strided_metrics() — MOT metrics on strided frame subsets ──────────
# Filters GT to only the strided frame IDs before computing metrics. Without
# this filter, unpredicted frames would count as all-miss, making MOTA ~ 0
# and MT always 0 — the metric would not reflect the tracking quality the
# device actually achieves on the frames it processes.

import motmetrics as mm
from benchmark.metrics import _build_accumulator


def compute_strided_metrics(
    gt_df: pd.DataFrame,
    pred_df: pd.DataFrame,
    strided_fids: list[int],
) -> dict:
    """MOT metrics computed on strided frame subsets only.

    Args:
        gt_df: Full GT DataFrame (all frames).
        pred_df: Predictions from strided_retrack() — only strided frames.
        strided_fids: List of original frame IDs in the strided subset.

    Returns:
        Dict with mota, idf1, num_switches, idsw_per_gt_track,
        mostly_tracked_ratio, n_gt_tracks, n_strided_frames.
    """
    # Filter GT to strided frames only
    gt_strided = gt_df[gt_df["frame_id"].isin(set(strided_fids))].copy()
    n_gt = gt_strided["track_id"].nunique()

    # Build accumulator and compute standard MOT metrics
    acc = _build_accumulator(gt_strided, pred_df)
    mh = mm.metrics.create()
    summary = mh.compute(
        acc,
        metrics=["mota", "idf1", "num_switches", "mostly_tracked", "mostly_lost"],
        name="seq",
    )

    num_switches = int(summary.loc["seq", "num_switches"])
    mostly_tracked = int(summary.loc["seq", "mostly_tracked"])

    return {
        "mota":                 float(summary.loc["seq", "mota"]),
        "idf1":                 float(summary.loc["seq", "idf1"]),
        "num_switches":         num_switches,
        "idsw_per_gt_track":    num_switches / n_gt if n_gt > 0 else float("nan"),
        "mostly_tracked_ratio": mostly_tracked / n_gt if n_gt > 0 else float("nan"),
        "mostly_lost":          int(summary.loc["seq", "mostly_lost"]),
        "n_gt_tracks":          n_gt,
        "n_strided_frames":     len(strided_fids),
    }


def parallel_map_tasks(tasks: list[tuple], worker_fn):
    """Run independent replay tasks in parallel when fork-based workers exist."""
    if not tasks:
        return []
    if not USE_PARALLEL:
        return [worker_fn(task) for task in tasks]

    ctx = mp.get_context(MP_START_METHOD)
    chunksize = max(1, len(tasks) // (N_WORKERS * 4))
    with ProcessPoolExecutor(max_workers=N_WORKERS, mp_context=ctx) as ex:
        return list(ex.map(worker_fn, tasks, chunksize=chunksize))


def run_sweep_task(task: tuple[str, int, int, int, str, str, int, str]) -> dict:
    model, imgsz, fps_level, stride, precision, buf_mode, track_buf, seq = task
    raw_df = det_cache[(seq, model, imgsz, precision)]
    pred_df, strided_fids = strided_retrack(raw_df, stride=stride, track_buffer=track_buf)
    metrics = compute_strided_metrics(gt_cache[seq], pred_df, strided_fids)
    return {
        "model": model,
        "imgsz": imgsz,
        "fps_level": fps_level,
        "stride": stride,
        "precision": precision,
        "buf_mode": buf_mode,
        "track_buffer": track_buf,
        "seq": seq,
        **metrics,
    }


def exact_quality_task(task: tuple[str, int, str, int]):
    model, imgsz, precision, eval_fps = task
    track_buffer = max(5, eval_fps)
    rows = []
    for seq in SEQUENCES:
        cache_key = (seq, model, imgsz, precision)
        if cache_key not in det_cache:
            continue
        pred_df, sampled_fids = retrack_at_target_fps(
            det_cache[cache_key],
            target_fps=eval_fps,
            track_buffer=track_buffer,
        )
        rows.append(compute_strided_metrics(gt_cache[seq], pred_df, sampled_fids))

    if not rows:
        return task, None

    exact_df = pd.DataFrame(rows)
    return task, {
        "mota": weighted_avg(exact_df, "mota"),
        "idsw_per_gt_track": weighted_avg(exact_df, "idsw_per_gt_track"),
        "mostly_tracked_ratio": weighted_avg(exact_df, "mostly_tracked_ratio"),
    }


# Validation: stride=1 metrics on the test key
_test_gt = gt_cache[_test_key[0]]
_test_metrics = compute_strided_metrics(_test_gt, _test_pred, _test_fids)
print(f"Stride=1 metrics for {_test_key}:")
for k, v in _test_metrics.items():
    print(f"  {k:>24s}: {v}")

In [ ]:
# ── Main sweep loop ───────────────────────────────────────────────────────────
# Iterate: model × imgsz × fps_level × precision × buf_mode × sequence.
# buf_mode='fixed' only at stride ∈ {15, 30} (sensitivity check).
# Skip missing cache entries. Collect results → results_df.
from itertools import product
import time

tasks = []
t0_sweep = time.perf_counter()

for model, imgsz in product(MODELS, RESOLUTIONS):
    for fps_level, stride in FPS_LEVELS.items():
        for precision in ["fp32", "int8", "trt"]:
            # Determine which buf_modes to run
            buf_modes = ["scaled"]
            if stride in (15, 30):
                buf_modes.append("fixed")

            for buf_mode in buf_modes:
                track_buf = get_track_buffer(stride, mode=buf_mode)

                for seq in SEQUENCES:
                    cache_key = (seq, model, imgsz, precision)
                    if cache_key not in det_cache:
                        continue
                    tasks.append((
                        model, imgsz, fps_level, stride,
                        precision, buf_mode, track_buf, seq,
                    ))

results = parallel_map_tasks(tasks, run_sweep_task)
elapsed = time.perf_counter() - t0_sweep
n_runs = len(results)
results_df = pd.DataFrame(results)

# Save raw results
results_csv = OUT_DIR / "quality_by_fps_level.csv"
results_df.to_csv(results_csv, index=False)

print(f"Completed {n_runs} re-tracking runs in {elapsed:.1f} s using {N_WORKERS if USE_PARALLEL else 1} worker(s)")
print(f"Saved to {results_csv}")
print(f"\nResults shape: {results_df.shape}")
print(results_df.head(10))

In [ ]:
# ── Effect decomposition analysis ─────────────────────────────────────────────
# Aggregate across sequences (weighted by GT track count) and compute the
# four quality effects from the plan.
#
# Effect B (temporal, FP32): fp32_stride1 − fp32_strided per FPS level
# Effect C (quantisation):   fp32_stride1 − int8_stride1
# Effect D (temporal, INT8): int8_stride1 − int8_strided per FPS level
# Effect A (resolution):     already exists from Experiment 1

# Weighted average across sequences for a given condition
def weighted_avg(df_slice, metric, weight_col="n_gt_tracks"):
    """GT-track-weighted average of a metric across sequences."""
    if df_slice.empty:
        return float("nan")
    return np.average(df_slice[metric], weights=df_slice[weight_col])


# Filter to scaled buffer mode only (primary results)
primary = results_df[results_df["buf_mode"] == "scaled"].copy()

# Pivot: per (model, imgsz, fps_level, precision) → weighted metrics
effect_rows = []
for model in MODELS:
    for imgsz in RESOLUTIONS:
        # FP32 stride=1 baseline (desktop offline quality)
        fp32_s1 = primary[
            (primary["model"] == model) &
            (primary["imgsz"] == imgsz) &
            (primary["precision"] == "fp32") &
            (primary["fps_level"] == 30)
        ]
        mt_fp32_ref = weighted_avg(fp32_s1, "mostly_tracked_ratio")
        idsw_fp32_ref = weighted_avg(fp32_s1, "idsw_per_gt_track")

        # INT8 stride=1 baseline (Hailo offline quality)
        int8_s1 = primary[
            (primary["model"] == model) &
            (primary["imgsz"] == imgsz) &
            (primary["precision"] == "int8") &
            (primary["fps_level"] == 30)
        ]
        mt_int8_ref = weighted_avg(int8_s1, "mostly_tracked_ratio")
        idsw_int8_ref = weighted_avg(int8_s1, "idsw_per_gt_track")

        # Effect C (quantisation): FP32 stride=1 − INT8 stride=1
        effect_c_mt = mt_fp32_ref - mt_int8_ref
        effect_c_idsw = idsw_int8_ref - idsw_fp32_ref  # positive = more switches = worse

        for fps_level, stride in FPS_LEVELS.items():
            # Effect B (temporal, FP32)
            fp32_strided = primary[
                (primary["model"] == model) &
                (primary["imgsz"] == imgsz) &
                (primary["precision"] == "fp32") &
                (primary["fps_level"] == fps_level)
            ]
            mt_fp32 = weighted_avg(fp32_strided, "mostly_tracked_ratio")
            idsw_fp32 = weighted_avg(fp32_strided, "idsw_per_gt_track")
            effect_b_mt = mt_fp32_ref - mt_fp32
            effect_b_idsw = idsw_fp32 - idsw_fp32_ref

            # Effect D (temporal, INT8)
            int8_strided = primary[
                (primary["model"] == model) &
                (primary["imgsz"] == imgsz) &
                (primary["precision"] == "int8") &
                (primary["fps_level"] == fps_level)
            ]
            mt_int8 = weighted_avg(int8_strided, "mostly_tracked_ratio")
            idsw_int8 = weighted_avg(int8_strided, "idsw_per_gt_track")
            effect_d_mt = mt_int8_ref - mt_int8
            effect_d_idsw = idsw_int8 - idsw_int8_ref

            effect_rows.append({
                "model": model, "imgsz": imgsz, "fps_level": fps_level,
                "stride": stride,
                # Weighted averages at this condition
                "mt_fp32": mt_fp32, "mt_int8": mt_int8,
                "idsw_fp32": idsw_fp32, "idsw_int8": idsw_int8,
                # Reference baselines (stride=1)
                "mt_fp32_ref": mt_fp32_ref, "mt_int8_ref": mt_int8_ref,
                "idsw_fp32_ref": idsw_fp32_ref, "idsw_int8_ref": idsw_int8_ref,
                # Effects
                "effect_b_mt": effect_b_mt, "effect_b_idsw": effect_b_idsw,
                "effect_c_mt": effect_c_mt, "effect_c_idsw": effect_c_idsw,
                "effect_d_mt": effect_d_mt, "effect_d_idsw": effect_d_idsw,
            })

effects_df = pd.DataFrame(effect_rows)
effects_csv = OUT_DIR / "effect_decomposition.csv"
effects_df.to_csv(effects_csv, index=False)

print(f"Effect decomposition saved to {effects_csv}")
print(f"\nEffect C (quantisation) — MT loss at stride=1, 640px:")
for model in MODELS:
    row = effects_df[(effects_df["model"] == model) &
                     (effects_df["imgsz"] == 640) &
                     (effects_df["fps_level"] == 30)].iloc[0]
    print(f"  {model}: MT gap = {row['effect_c_mt']:+.4f}, "
          f"IDSW gap = {row['effect_c_idsw']:+.4f}")

print(f"\nEffect B (temporal, FP32) — MT loss at 640px, selected strides:")
for model in MODELS:
    for fps in [15, 5, 1]:
        row = effects_df[(effects_df["model"] == model) &
                         (effects_df["imgsz"] == 640) &
                         (effects_df["fps_level"] == fps)].iloc[0]
        print(f"  {model} @ {fps} FPS: MT loss = {row['effect_b_mt']:+.4f}")

In [ ]:
# ── Degradation curve plot — MT vs FPS level ──────────────────────────────────
# Three panels (N / S / M), one per model variant at 640px resolution.
# Solid line:  FP32 temporal degradation (effect B)
# Dashed line: Hailo INT8 temporal degradation (effect D)
# Markers:     each device's actual FPS position
# Horizontal reference: stride=1 offline quality

import matplotlib; matplotlib.rc_file("../matplotlibrc")
import matplotlib.pyplot as plt

MODEL_LABELS = {"yolo26n": "YOLO26-N", "yolo26s": "YOLO26-S", "yolo26m": "YOLO26-M"}
DEVICE_MARKERS = {
    dev_id: (meta["marker"], meta["color"], meta["label"])
    for dev_id, meta in PROFILES.items()
    if dev_id in DEVICE_MAP
}

eff_640 = effects_df[effects_df["imgsz"] == 640].copy()

fig, axes = plt.subplots(1, 3, figsize=(12, 4), sharey=True, constrained_layout=True)

for ax, model in zip(axes, MODELS):
    sub = eff_640[eff_640["model"] == model].sort_values("fps_level")

    # FP32 temporal degradation curve
    ax.plot(sub["fps_level"], sub["mt_fp32"], "s-", color="#1f77b4",
            linewidth=1.8, markersize=5, label="FP32 (effect B)")

    # INT8 temporal degradation curve
    ax.plot(sub["fps_level"], sub["mt_int8"], "D--", color="#ff7f0e",
            linewidth=1.8, markersize=5, label="INT8 (effect D)")

    # FP32 stride=1 reference line
    ref_fp32 = sub["mt_fp32_ref"].iloc[0]
    ref_int8 = sub["mt_int8_ref"].iloc[0]
    ax.axhline(ref_fp32, color="#1f77b4", linewidth=0.8, alpha=0.4, linestyle=":")
    ax.axhline(ref_int8, color="#ff7f0e", linewidth=0.8, alpha=0.4, linestyle=":")

    # Device markers on the FP32 curve
    variant_letter = model.replace("yolo26", "").upper()
    for dev_id, (marker, color, label) in DEVICE_MARKERS.items():
        if variant_letter not in DEVICE_MAP.get(dev_id, {}):
            continue
        actual_fps, det_src = DEVICE_MAP[dev_id][variant_letter]
        fps_lvl = assign_fps_level(actual_fps)

        # Pick the right curve (FP32 or INT8) or a backend-specific marker value.
        if det_src == "fp32":
            mt_val = sub[sub["fps_level"] == fps_lvl]["mt_fp32"].values
        elif det_src == "int8":
            mt_val = sub[sub["fps_level"] == fps_lvl]["mt_int8"].values
        else:
            trt_slice = primary[
                (primary["model"] == model) &
                (primary["imgsz"] == 640) &
                (primary["precision"] == det_src) &
                (primary["fps_level"] == fps_lvl)
            ]
            mt_val = np.array([weighted_avg(trt_slice, "mostly_tracked_ratio")]) if not trt_slice.empty else np.array([])

        if len(mt_val) > 0:
            ax.plot(fps_lvl, mt_val[0], marker=marker, color=color,
                    markersize=9, markeredgecolor="black", markeredgewidth=0.5,
                    label=label, zorder=5)

    ax.set_title(MODEL_LABELS[model], fontsize=12, fontweight="bold")
    ax.set_xscale("log", base=2)
    ax.set_xticks(sorted(FPS_LEVELS.keys()))
    ax.set_xticklabels([str(x) for x in sorted(FPS_LEVELS.keys())])
    ax.grid(axis="y", linewidth=0.4, alpha=0.4)
    ax.set_xlim(0.8, 35)
    ax.invert_xaxis()  # high FPS (good) on left, low FPS (bad) on right

# Shared x-axis label on the center panel only (avoids overlap with legend)
axes[1].set_xlabel("FPS level", fontsize=10)
axes[0].set_ylabel("Mostly Tracked ratio\n(higher = better)", fontsize=10)

# Deduplicated legend — placed below with enough clearance from axis labels
handles, labels = axes[0].get_legend_handles_labels()
by_label = dict(zip(labels, handles))
fig.legend(by_label.values(), by_label.keys(), loc="lower center",
           ncol=4, fontsize=9, bbox_to_anchor=(0.5, -0.14))

fig_path = RESULTS_RAW.parent / "figures" / "fig_temporal_degradation.pdf"
fig_path.parent.mkdir(parents=True, exist_ok=True)
plt.savefig(fig_path, bbox_inches="tight", dpi=150)
plt.show()
print(f"Saved → {fig_path}")

In [ ]:
# ── track_buffer sensitivity report ───────────────────────────────────────────
# At stride=15 (2 FPS) and stride=30 (1 FPS), compare scaled vs fixed buffer.
# Threshold: < 5% absolute MT delta = robustness finding.

sensitivity_rows = []

for stride_check in [15, 30]:
    fps_lvl = {15: 2, 30: 1}[stride_check]
    for model in MODELS:
        for imgsz in RESOLUTIONS:
            for precision in ["fp32", "int8"]:
                scaled = results_df[
                    (results_df["model"] == model) &
                    (results_df["imgsz"] == imgsz) &
                    (results_df["precision"] == precision) &
                    (results_df["stride"] == stride_check) &
                    (results_df["buf_mode"] == "scaled")
                ]
                fixed = results_df[
                    (results_df["model"] == model) &
                    (results_df["imgsz"] == imgsz) &
                    (results_df["precision"] == precision) &
                    (results_df["stride"] == stride_check) &
                    (results_df["buf_mode"] == "fixed")
                ]
                if scaled.empty or fixed.empty:
                    continue

                # Weighted averages across sequences
                mt_scaled = weighted_avg(scaled, "mostly_tracked_ratio")
                mt_fixed = weighted_avg(fixed, "mostly_tracked_ratio")
                idsw_scaled = weighted_avg(scaled, "idsw_per_gt_track")
                idsw_fixed = weighted_avg(fixed, "idsw_per_gt_track")

                sensitivity_rows.append({
                    "model": model, "imgsz": imgsz, "precision": precision,
                    "stride": stride_check, "fps_level": fps_lvl,
                    "mt_scaled": mt_scaled, "mt_fixed": mt_fixed,
                    "mt_delta": mt_fixed - mt_scaled,
                    "idsw_scaled": idsw_scaled, "idsw_fixed": idsw_fixed,
                    "idsw_delta": idsw_fixed - idsw_scaled,
                })

sens_df = pd.DataFrame(sensitivity_rows)
sens_csv = OUT_DIR / "track_buffer_sensitivity.csv"
sens_df.to_csv(sens_csv, index=False)

print("track_buffer sensitivity: scaled vs fixed (positive delta = fixed is higher)")
print(f"{'model':>8s} {'imgsz':>5s} {'prec':>5s} {'stride':>6s} "
      f"{'MT_sc':>7s} {'MT_fx':>7s} {'MT_d':>7s} "
      f"{'IDSW_sc':>8s} {'IDSW_fx':>8s} {'IDSW_d':>8s}")
for _, r in sens_df.iterrows():
    print(f"{r['model']:>8s} {r['imgsz']:>5.0f} {r['precision']:>5s} {r['stride']:>6.0f} "
          f"{r['mt_scaled']:>7.4f} {r['mt_fixed']:>7.4f} {r['mt_delta']:>+7.4f} "
          f"{r['idsw_scaled']:>8.4f} {r['idsw_fixed']:>8.4f} {r['idsw_delta']:>+8.4f}")

# Summary verdict
max_mt_delta = sens_df["mt_delta"].abs().max()
print(f"\nMax |MT delta|: {max_mt_delta:.4f}")
if max_mt_delta < 0.05:
    print("VERDICT: track_buffer scaling is robust — < 5% MT delta at all conditions.")
else:
    print("VERDICT: track_buffer sensitivity detected — discuss in methodology section.")

In [ ]:
# ── Table I assembly ──────────────────────────────────────────────────────────
# Reference rows: FP32 stride=1 quality per model variant (dashes in HW columns).
# Device rows: real-time quality at device operational FPS, replayed at the
# floor(actual_fps) integer cadence (capped at 30 FPS). Hardware metrics come
# from existing profiling data.
#
# Quality columns: MOTA, IDSW/GT, MT — from weighted averages across sequences.

# Hardware summaries are loaded from profiling outputs so rerunning any device
# profile automatically refreshes the temporal table.
import re

PROFILE_DIR = RESULTS_RAW.parent / "profiling"


def profile_det_source(dev_id: str) -> str | None:
    variant_map = DEVICE_MAP.get(dev_id, {})
    if not variant_map:
        return None
    return next(iter(variant_map.values()))[1]


def profile_display_label(dev_id: str) -> str:
    suffix = " †" if profile_det_source(dev_id) == "int8" else ""
    return f"{PROFILES[dev_id]['label']}{suffix}"


def model_variant(model_name: str) -> str | None:
    match = re.search(r"yolo26([nsm])", str(model_name))
    return match.group(1).upper() if match else None


def load_hardware_data(profile_dir: Path) -> dict[tuple[str, str, int], tuple[float | None, float | None, float | None, float | None]]:
    hardware = {}

    for dev_id in PROFILES:
        profiling_path = profile_dir / f"table1_profiling_{dev_id}.csv"
        power_path = profile_dir / f"table1_power_{dev_id}.csv"

        prof_summary = pd.DataFrame(columns=["variant", "imgsz", "fps", "p95_ms", "mem_mb"])
        if profiling_path.exists():
            prof_df = pd.read_csv(profiling_path)
            # Normalise p95 column name: edge CSVs use "e2e_p95_ms", desktop uses "p95_ms"
            if "e2e_p95_ms" in prof_df.columns and "p95_ms" not in prof_df.columns:
                prof_df = prof_df.rename(columns={"e2e_p95_ms": "p95_ms"})
            prof_df["variant"] = prof_df["model"].map(model_variant)
            prof_df = prof_df[prof_df["variant"].notna()].copy()
            prof_summary = prof_df.groupby(["variant", "imgsz"], as_index=False).agg(
                fps=("fps", "mean"),
                p95_ms=("p95_ms", "mean"),
                mem_mb=("peak_mem_mb", "mean"),
            )

        pwr_summary = pd.DataFrame(columns=["variant", "imgsz", "power_w"])
        if power_path.exists():
            pwr_df = pd.read_csv(power_path)
            pwr_df["variant"] = pwr_df["model"].map(model_variant)
            pwr_df = pwr_df[pwr_df["variant"].notna()].copy()
            if "mean_W" in pwr_df.columns:
                pwr_summary = pwr_df.groupby(["variant", "imgsz"], as_index=False).agg(
                    power_w=("mean_W", "mean"),
                )

        merged = prof_summary.merge(pwr_summary, how="outer", on=["variant", "imgsz"])
        for _, row in merged.iterrows():
            hardware[(dev_id, row["variant"], int(row["imgsz"]))] = (
                float(row["fps"]) if pd.notna(row.get("fps")) else None,
                float(row["p95_ms"]) if pd.notna(row.get("p95_ms")) else None,
                float(row["mem_mb"]) if pd.notna(row.get("mem_mb")) else None,
                float(row["power_w"]) if pd.notna(row.get("power_w")) else None,
            )

    return hardware


HARDWARE_DATA = load_hardware_data(PROFILE_DIR)

def table_eval_fps(actual_fps: float | None, source_fps: int = 30) -> int | None:
    """Conservative per-device replay rate used for paper-table quality rows."""
    if actual_fps is None or pd.isna(actual_fps):
        return None
    return max(1, min(source_fps, int(np.floor(actual_fps))))


exact_tasks = set()
for model in MODELS:
    variant = model.replace("yolo26", "").upper()
    for imgsz in RESOLUTIONS:
        for dev_id in PROFILES:
            if variant not in DEVICE_MAP.get(dev_id, {}):
                continue
            default_fps, det_src = DEVICE_MAP[dev_id][variant]
            hw = HARDWARE_DATA.get((dev_id, variant, imgsz), (None, None, None, None))
            actual_fps = hw[0] if hw[0] is not None else default_fps
            eval_fps = table_eval_fps(actual_fps)
            if eval_fps is not None:
                exact_tasks.add((model, imgsz, det_src, eval_fps))

device_quality_cache = {
    key: value
    for key, value in parallel_map_tasks(sorted(exact_tasks), exact_quality_task)
    if value is not None
}

# Build table rows
table_rows = []

for model in MODELS:
    variant = model.replace("yolo26", "").upper()

    for imgsz in RESOLUTIONS:
        # Reference row: FP32 stride=1 offline quality
        ref_slice = primary[
            (primary["model"] == model) &
            (primary["imgsz"] == imgsz) &
            (primary["precision"] == "fp32") &
            (primary["fps_level"] == 30)
        ]
        if not ref_slice.empty:
            table_rows.append({
                "device": "Ref. (FP32, 30 FPS)*",
                "variant": variant, "imgsz": imgsz,
                "fps": None, "p95_ms": None, "mem_mb": None, "power_w": None,
                "mota": weighted_avg(ref_slice, "mota"),
                "idsw_per_gt_track": weighted_avg(ref_slice, "idsw_per_gt_track"),
                "mostly_tracked_ratio": weighted_avg(ref_slice, "mostly_tracked_ratio"),
                "row_type": "reference",
            })

        # Device rows: real-time quality at operational FPS
        for dev_id in PROFILES:
            if variant not in DEVICE_MAP.get(dev_id, {}):
                continue
            default_fps, det_src = DEVICE_MAP[dev_id][variant]
            hw = HARDWARE_DATA.get((dev_id, variant, imgsz), (None, None, None, None))
            actual_fps = hw[0] if hw[0] is not None else default_fps
            eval_fps = table_eval_fps(actual_fps)
            if eval_fps is None:
                continue

            exact_key = (model, imgsz, det_src, eval_fps)
            exact_quality = device_quality_cache.get(exact_key)
            if exact_quality is None:
                continue

            table_rows.append({
                "device": profile_display_label(dev_id),
                "variant": variant, "imgsz": imgsz,
                "eval_fps": eval_fps,
                "fps": hw[0], "p95_ms": hw[1], "mem_mb": hw[2], "power_w": hw[3],
                "mota": exact_quality["mota"],
                "idsw_per_gt_track": exact_quality["idsw_per_gt_track"],
                "mostly_tracked_ratio": exact_quality["mostly_tracked_ratio"],
                "row_type": "device",
            })

table_df = pd.DataFrame(table_rows)

# Save CSV
table_csv = OUT_DIR / "table1_temporal.csv"
table_df.to_csv(table_csv, index=False)

# Display — wide view with both resolutions
pd.set_option("display.max_columns", 20)
pd.set_option("display.width", 240)
print("Table I — Temporal Degradation:")
display_cols = [
    "device", "variant", "imgsz", "fps", "eval_fps", "mem_mb", "power_w",
    "mota", "idsw_per_gt_track", "mostly_tracked_ratio",
]
t_disp = table_df[display_cols].copy()
for col in ["mota", "idsw_per_gt_track", "mostly_tracked_ratio"]:
    t_disp[col] = t_disp[col].map(lambda x: f"{x:.4f}" if pd.notna(x) else "\u2014")
for col in ["fps", "mem_mb", "power_w"]:
    t_disp[col] = t_disp[col].map(lambda x: f"{x:.1f}" if pd.notna(x) else "\u2014")
t_disp["eval_fps"] = t_disp["eval_fps"].map(lambda x: f"{int(x)}" if pd.notna(x) else "\u2014")
print(t_disp.to_string(index=False))

# ── Paper-ready LaTeX ─────────────────────────────────────────────────────────
tex_path = OUT_DIR / "table1_temporal.tex"

# Quality lookups keyed by (device_label, variant) for each resolution
quality_lookup_640 = {
    (row["device"], row["variant"]): row
    for _, row in table_df[(table_df["imgsz"] == 640) & (table_df["row_type"] == "device")].iterrows()
}
quality_lookup_576 = {
    (row["device"], row["variant"]): row
    for _, row in table_df[(table_df["imgsz"] == 576) & (table_df["row_type"] == "device")].iterrows()
}
reference_lookup_640 = {
    row["variant"]: row
    for _, row in table_df[(table_df["imgsz"] == 640) & (table_df["row_type"] == "reference")].iterrows()
}
reference_lookup_576 = {
    row["variant"]: row
    for _, row in table_df[(table_df["imgsz"] == 576) & (table_df["row_type"] == "reference")].iterrows()
}

TABLE_BLOCKS = [(r"\multirow{3}{*}{\makecell[l]{Ref.\\FP32\\30 FPS\tnote{*}}}", "reference")]
for dev_id, meta in PROFILES.items():
    TABLE_BLOCKS.append((
        rf"\multirow{{3}}{{*}}{{{meta['table_tex']}}}",
        profile_display_label(dev_id) if dev_id in DEVICE_MAP else None,
    ))


def lookup_table_row(device_label: str, variant: str, imgsz: int, row_type: str = "device"):
    sub = table_df[
        (table_df["device"] == device_label) &
        (table_df["variant"] == variant) &
        (table_df["imgsz"] == imgsz) &
        (table_df["row_type"] == row_type)
    ]
    return sub.iloc[0] if not sub.empty else None


def fmt_hw_value(value, kind: str) -> str:
    if value is None or pd.isna(value):
        return "---"
    if kind == "mem":
        return f"{value:.0f}"
    return f"{value:.1f}"


def fmt_mota(x):
    s = f"{x:.3f}"
    return s[1:] if s.startswith("0") else s

def fmt_mt(x):
    s = f"{x:.3f}"
    return s[1:] if s.startswith("0") else s

def fmt_idsw(x):
    return f"{x:.2f}"

def fmt_delta_cell(value, ref_value, kind):
    if kind == "mota":
        base = fmt_mota(value)
    elif kind == "mt":
        base = fmt_mt(value)
    else:
        base = fmt_idsw(value)

    pct = round((value / ref_value - 1.0) * 100.0)
    return rf"\makecell[r]{{{base}\\[-2pt]{{\scriptsize({pct:+d}\%)}}}}"


# Build per-block row data: each row is a dict with all 12 data columns
paper_rows = []
for device_tex, lookup_device in TABLE_BLOCKS:
    rows = []
    for variant in ["N", "S", "M"]:
        row_data = {"device_tex": device_tex, "lookup_device": lookup_device, "variant": variant}

        if lookup_device == "reference" or lookup_device is None:
            rows.append(row_data)
            continue

        row640 = lookup_table_row(lookup_device, variant, 640)
        row576 = lookup_table_row(lookup_device, variant, 576)
        row_data["raw_fps640"] = row640["fps"] if row640 is not None else None
        row_data["raw_fps576"] = row576["fps"] if row576 is not None else None
        row_data["fps640"] = fmt_hw_value(row640["fps"] if row640 is not None else None, "fps")
        row_data["fps576"] = fmt_hw_value(row576["fps"] if row576 is not None else None, "fps")
        # Single mem/pwr columns from 640 px (invariant across resolutions)
        row_data["mem"] = fmt_hw_value(row640["mem_mb"] if row640 is not None else None, "mem")
        row_data["pwr"] = fmt_hw_value(row640["power_w"] if row640 is not None else None, "pwr")
        rows.append(row_data)

    paper_rows.append(rows)

# Column spec: lc rr r r | rrr rrr  (12 data columns)
latex_lines = [
    r"\begin{table}[htp]",
    r"\caption{Per-device, per-variant profiling and real-time tracking quality at two resolutions.}",
    r"\label{tab:profiling}",
    r"\centering",
    r"\begin{threeparttable}",
    r"\setlength{\tabcolsep}{1.5pt}",
    r"\begin{tabular}{@{}lc rr r r rrr rrr@{}}",
    r"\toprule",
    r"& &",
    r"  \multicolumn{2}{c}{FPS} &",
    r"  &",
    r"  &",
    r"  \multicolumn{3}{c}{Quality @ 640\,px} &",
    r"  \multicolumn{3}{c}{Quality @ 576\,px} \\",
    r"\cmidrule(lr){3-4}\cmidrule(lr){7-9}\cmidrule(lr){10-12}",
    r"Device & V &",
    r"  640 & 576 &",
    r"  \makecell{Mem\\(MB)\tnote{$\ddagger$}} &",
    r"  \makecell{Pwr\\(W)\tnote{$\ddagger$}} &",
    "  \\makecell{MO-\\\\TA\\\\[-1pt]{\\tiny$\\uparrow$}} & \\makecell{IDSW\\\\$/\\!$\\,GT\\\\[-1pt]{\\tiny$\\downarrow$}} & \\makecell{MT\\\\[-1pt]{\\tiny$\\uparrow$}} &",
    "  \\makecell{MO-\\\\TA\\\\[-1pt]{\\tiny$\\uparrow$}} & \\makecell{IDSW\\\\$/\\!$\\,GT\\\\[-1pt]{\\tiny$\\downarrow$}} & \\makecell{MT\\\\[-1pt]{\\tiny$\\uparrow$}} \\\\",
    r"\midrule",
]

for block_idx, rows in enumerate(paper_rows):
    for row_idx, rd in enumerate(rows):
        device_tex = rd["device_tex"]
        variant = rd["variant"]
        lookup_device = rd["lookup_device"]
        prefix = f"{device_tex}\n  " if row_idx == 0 else "  "

        if lookup_device is None:
            latex_lines.append(
                prefix + f"& {variant} & --- & --- & --- & --- & --- & --- & --- & --- & --- & --- \\\\"
            )
            continue

        if lookup_device == "reference":
            q640 = reference_lookup_640.get(variant)
            q576 = reference_lookup_576.get(variant)
            m640 = fmt_mota(q640["mota"]) if q640 is not None else "---"
            i640 = fmt_idsw(q640["idsw_per_gt_track"]) if q640 is not None else "---"
            t640 = fmt_mt(q640["mostly_tracked_ratio"]) if q640 is not None else "---"
            m576 = fmt_mota(q576["mota"]) if q576 is not None else "---"
            i576 = fmt_idsw(q576["idsw_per_gt_track"]) if q576 is not None else "---"
            t576 = fmt_mt(q576["mostly_tracked_ratio"]) if q576 is not None else "---"
            latex_lines.append(
                prefix + f"& {variant} & \\multicolumn{{4}}{{c}}{{\\scriptsize offline reference}} & {m640} & {i640} & {t640} & {m576} & {i576} & {t576} \\\\"
            )
            continue

        fps640 = rd.get("fps640", "---")
        fps576 = rd.get("fps576", "---")
        mem = rd.get("mem", "---")
        pwr = rd.get("pwr", "---")

        # Quality at 640 — suppress when actual FPS < 1 (sub-real-time)
        raw_fps_640 = rd.get("raw_fps640")
        qk640 = quality_lookup_640.get((lookup_device, variant))
        qr640 = reference_lookup_640.get(variant)
        if raw_fps_640 is not None and raw_fps_640 < 1.0:
            m640 = i640 = t640 = "n.a."
        elif qk640 is not None and qr640 is not None:
            m640 = fmt_delta_cell(qk640["mota"], qr640["mota"], "mota")
            i640 = fmt_delta_cell(qk640["idsw_per_gt_track"], qr640["idsw_per_gt_track"], "idsw")
            t640 = fmt_delta_cell(qk640["mostly_tracked_ratio"], qr640["mostly_tracked_ratio"], "mt")
        else:
            m640 = i640 = t640 = "---"

        # Quality at 576 — suppress when actual FPS < 1 (sub-real-time)
        raw_fps_576 = rd.get("raw_fps576")
        qk576 = quality_lookup_576.get((lookup_device, variant))
        qr576 = reference_lookup_576.get(variant)
        if raw_fps_576 is not None and raw_fps_576 < 1.0:
            m576 = i576 = t576 = "n.a."
        elif qk576 is not None and qr576 is not None:
            m576 = fmt_delta_cell(qk576["mota"], qr576["mota"], "mota")
            i576 = fmt_delta_cell(qk576["idsw_per_gt_track"], qr576["idsw_per_gt_track"], "idsw")
            t576 = fmt_delta_cell(qk576["mostly_tracked_ratio"], qr576["mostly_tracked_ratio"], "mt")
        else:
            m576 = i576 = t576 = "---"

        latex_lines.append(
            prefix + f"& {variant} & {fps640} & {fps576} & {mem} & {pwr} & {m640} & {i640} & {t640} & {m576} & {i576} & {t576} \\\\"
        )
    latex_lines.append(r"\midrule" if block_idx < len(paper_rows) - 1 else r"\bottomrule")

latex_lines.extend([
    r"\end{tabular}",
    r"\begin{tablenotes}\footnotesize",
    r"  \item[$*$] Reference rows report offline sequential FP32 quality at 30 FPS on the desktop pipeline.",
    r"  \item[$\ddagger$] Memory and power are measured at 640\,px; both are invariant to resolution (see Section~\ref{sec:profiling}).",
    r"  \item[] Parenthetical percentages are relative to the matching FP32 reference row at the same resolution.",
    r"  \item[] Device-row quality columns report real-time temporal replay estimates at $\lfloor\text{FPS}\rfloor$, capped at 30.",
    r"  \item[$\dagger$] INT8 quantization via HailoRT.",
    r"  \item[] n.a.\,= not applicable; device throughput ${<}\,1$\,FPS makes real-time tracking meaningless.",
    r"\end{tablenotes}",
    r"\end{threeparttable}",
    r"\end{table}",
])

tex_path.write_text("\n".join(latex_lines) + "\n")
print(f"\nSaved CSV  \u2192 {table_csv}")
print(f"Saved LaTeX \u2192 {tex_path}")